# Layerwise Logits EDA And Feature AUC

Static analysis notebook for exported layerwise, attention, and substep readouts.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

In [ ]:
LETTERS = ["A", "B", "C", "D", "E"]
RUN_DIR = Path("..") / ".." / "data" / "20260426-183542_Qwen-Qwen2.5-3B-Instruct_csqa_base_layerwise"
MODEL_SPLIT_PREFIX = ""

In [ ]:
examples_df = pd.read_parquet(RUN_DIR / "examples.parquet")
clean_final_df = pd.read_parquet(RUN_DIR / "clean_final_outputs.parquet")
clean_topk_df = pd.read_parquet(RUN_DIR / "clean_final_topk.parquet")
layerwise_df = pd.read_parquet(RUN_DIR / "layerwise_outputs.parquet")
tuned_lens_history_df = pd.read_parquet(RUN_DIR / "tuned_lens_training_history.parquet")

model_clean_final_path = RUN_DIR / f"{MODEL_SPLIT_PREFIX}clean_final_outputs.parquet"
model_layerwise_path = RUN_DIR / f"{MODEL_SPLIT_PREFIX}layerwise_outputs.parquet"
model_attention_path = RUN_DIR / f"{MODEL_SPLIT_PREFIX}attention_outputs.parquet"

attention_path = RUN_DIR / "attention_outputs.parquet"
substep_path = RUN_DIR / "substep_outputs.parquet"

attention_df = pd.read_parquet(attention_path)
substep_df = pd.read_parquet(substep_path)
model_clean_final_df = pd.read_parquet(model_clean_final_path)
model_layerwise_df = pd.read_parquet(model_layerwise_path)
model_attention_df = pd.read_parquet(model_attention_path)

with open(RUN_DIR / "run_config.json", "r", encoding="utf-8") as f:
    run_config = json.load(f)

In [ ]:
clean_small_df = clean_final_df[[
    "example_id",
    "clean_is_correct",
    "clean_predicted_choice_idx",
    "clean_predicted_choice_letter",
    "clean_true_answer_rank_within_choices",
]].copy()

trace_bucket_df = clean_small_df[["example_id", "clean_true_answer_rank_within_choices"]].copy()
trace_bucket_df["trace_bucket"] = np.where(
    trace_bucket_df["clean_true_answer_rank_within_choices"] == 1,
    "correct",
    np.where(trace_bucket_df["clean_true_answer_rank_within_choices"] == 2, "near_miss", "hard_miss"),
)

bucket_order = ["correct", "near_miss", "hard_miss"]
bucket_palette = {
    "correct": "#2a9d8f",
    "near_miss": "#e9c46a",
    "hard_miss": "#e76f51",
}

analysis_layer_df = (
    layerwise_df
    .merge(clean_small_df, on="example_id", how="left", validate="many_to_one")
    .merge(trace_bucket_df[["example_id", "trace_bucket"]], on="example_id", how="left", validate="many_to_one")
    .copy()
)
analysis_layer_df["final_error"] = (~analysis_layer_df["clean_is_correct"]).astype(int)
analysis_layer_df["final_outcome"] = np.where(analysis_layer_df["clean_is_correct"], "correct", "incorrect")
analysis_layer_df["max_choice_logit"] = analysis_layer_df[["logit_A", "logit_B", "logit_C", "logit_D", "logit_E"]].max(axis=1)
analysis_layer_df["best_choice_minus_best_non_choice_logit"] = (
    analysis_layer_df["max_choice_logit"] - analysis_layer_df["best_non_choice_logit"]
)
analysis_layer_df["prediction_agreement_with_final"] = (
    analysis_layer_df["predicted_choice_idx"] == analysis_layer_df["clean_predicted_choice_idx"]
).astype(int)

method_order = [m for m in ["direct_readout", "tuned_lens"] if m in analysis_layer_df["readout_method"].unique()]

analysis_attention_df = (
    attention_df
    .merge(clean_small_df, on="example_id", how="left", validate="many_to_one")
    .merge(trace_bucket_df[["example_id", "trace_bucket"]], on="example_id", how="left", validate="many_to_one")
    .copy()
)
analysis_attention_df["final_error"] = (~analysis_attention_df["clean_is_correct"]).astype(int)
analysis_attention_df["final_outcome"] = np.where(analysis_attention_df["clean_is_correct"], "correct", "incorrect")
analysis_attention_df["prediction_agreement_with_final"] = (
    analysis_attention_df["attention_choice_predicted_choice_idx"] == analysis_attention_df["clean_predicted_choice_idx"]
).astype(int)

analysis_substep_df = (
    substep_df
    .merge(clean_small_df, on="example_id", how="left", validate="many_to_one")
    .merge(trace_bucket_df[["example_id", "trace_bucket"]], on="example_id", how="left", validate="many_to_one")
    .copy()
)
analysis_substep_df["final_error"] = (~analysis_substep_df["clean_is_correct"]).astype(int)
analysis_substep_df["final_outcome"] = np.where(analysis_substep_df["clean_is_correct"], "correct", "incorrect")
analysis_substep_df["prediction_agreement_with_final"] = (
    analysis_substep_df["predicted_choice_idx"] == analysis_substep_df["clean_predicted_choice_idx"]
).astype(int)

model_clean_small_df = model_clean_final_df[[
    "example_id",
    "clean_is_correct",
    "clean_predicted_choice_idx",
    "clean_predicted_choice_letter",
]].copy()

model_analysis_layer_df = (
    model_layerwise_df
    .merge(model_clean_small_df, on="example_id", how="left", validate="many_to_one")
    .copy()
)
model_analysis_layer_df["final_error"] = (~model_analysis_layer_df["clean_is_correct"]).astype(int)
model_analysis_layer_df["max_choice_logit"] = model_analysis_layer_df[["logit_A", "logit_B", "logit_C", "logit_D", "logit_E"]].max(axis=1)
model_analysis_layer_df["best_choice_minus_best_non_choice_logit"] = (
    model_analysis_layer_df["max_choice_logit"] - model_analysis_layer_df["best_non_choice_logit"]
)

model_attention_analysis_df = (
    model_attention_df
    .merge(model_clean_small_df, on="example_id", how="left", validate="many_to_one")
    .copy()
)
model_attention_analysis_df["final_error"] = (~model_attention_analysis_df["clean_is_correct"]).astype(int)

## EDA

In [ ]:
cm_df = clean_final_df.copy()

cm = pd.crosstab(
    cm_df["true_choice_letter"],
    cm_df["clean_predicted_choice_letter"],
).reindex(index=LETTERS, columns=LETTERS, fill_value=0)

bucket_counts = (
    trace_bucket_df["trace_bucket"]
    .value_counts()
    .reindex(bucket_order)
    .rename_axis("trace_bucket")
    .reset_index(name="count")
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Final Output Confusion Matrix")
axes[0].set_xlabel("Predicted letter")
axes[0].set_ylabel("True letter")

sns.barplot(data=bucket_counts, x="trace_bucket", y="count", order=bucket_order, palette=bucket_palette, ax=axes[1])
axes[1].set_title("Trace Bucket Counts")
axes[1].set_xlabel("Trace bucket")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
final_tuned_df = analysis_layer_df.loc[
    analysis_layer_df["readout_method"].eq("tuned_lens")
    & analysis_layer_df["layer_number"].eq(run_config["num_layers"])
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=final_tuned_df,
    x="answer_choice_entropy_normalized",
    hue="trace_bucket",
    hue_order=bucket_order,
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    palette=bucket_palette,
    ax=axes[0],
)
axes[0].set_title("Final Choice Entropy Histogram")
axes[0].set_xlabel("Answer choice entropy")
axes[0].set_ylabel("Density")

sns.histplot(
    data=final_tuned_df,
    x="answer_choice_top1_top2_logit_gap",
    hue="trace_bucket",
    hue_order=bucket_order,
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    palette=bucket_palette,
    ax=axes[1],
)
axes[1].set_title("Final Top1-Top2 Gap Histogram")
axes[1].set_xlabel("Top1-top2 logit gap")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
def plot_layer_canvas(frame, method_col, metrics, titles, fig_title, group_col="trace_bucket", group_order=None, palette=None):
    methods = frame[method_col].dropna().drop_duplicates().tolist()
    fig, axes = plt.subplots(
        len(methods),
        len(metrics),
        figsize=(5.5 * len(metrics), 4.2 * len(methods)),
        sharex=True,
    )

    if len(methods) == 1 and len(metrics) == 1:
        axes = np.array([[axes]])
    elif len(methods) == 1:
        axes = np.array([axes])
    elif len(metrics) == 1:
        axes = axes[:, None]

    for r, method in enumerate(methods):
        part = frame.loc[frame[method_col].eq(method)]
        for c, (metric, title) in enumerate(zip(metrics, titles)):
            ax = axes[r, c]
            sns.lineplot(
                data=part,
                x="layer_number",
                y=metric,
                hue=group_col,
                hue_order=group_order,
                palette=palette,
                estimator="mean",
                errorbar=None,
                marker="o",
                ax=ax,
                legend=(r == 0 and c == 0),
            )
            ax.set_title(f"{method}: {title}")
            ax.set_xlabel("Layer")
            ax.set_ylabel("Mean value")
            if not (r == 0 and c == 0) and ax.legend_ is not None:
                ax.legend_.remove()

    plt.suptitle(fig_title, y=1.02, fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_layer_canvas(
    frame=analysis_layer_df,
    method_col="readout_method",
    metrics=[
        "full_vocab_entropy_normalized",
        "answer_choice_entropy_normalized",
        "answer_choice_top1_top2_logit_gap",
        "choice_kl_to_final",
        "best_choice_minus_best_non_choice_logit",
    ],
    titles=[
        "Full Vocabulary Entropy",
        "Answer Choice Entropy",
        "Top1-Top2 Logit Gap",
        "KL To Final Choice Distribution",
        "Best Choice Minus Best Non-Choice",
    ],
    fig_title="Layerwise Readout Features",
    group_col="trace_bucket",
    group_order=bucket_order,
    palette=bucket_palette,
)

In [ ]:
attention_plot_df = analysis_attention_df.copy()
attention_plot_df["attention_method"] = "decision_position_attention"

plot_layer_canvas(
    frame=attention_plot_df,
    method_col="attention_method",
    metrics=[
        "mean_head_renyi2_entropy_normalized",
        "aggregated_choice_attention_entropy_normalized",
        "aggregated_choice_attention_top1_top2_probability_gap",
        "prediction_agreement_with_final",
    ],
    titles=[
        "Mean Head 2-Renyi Entropy",
        "Aggregated Choice Attention Entropy",
        "Aggregated Choice Attention Top1-Top2 Gap",
        "Agreement With Final Prediction",
    ],
    fig_title="Attention Features",
    group_col="trace_bucket",
    group_order=bucket_order,
    palette=bucket_palette,
)

In [ ]:
substep_order = ["pre_attn", "post_attn", "post_mlp"]
metric_cols = [
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
    "best_choice_minus_best_non_choice_logit",
    "choice_kl_to_final",
]
metric_titles = [
    "Choice Entropy",
    "Top1-Top2 Gap",
    "Best Choice Minus Best Non-Choice",
    "KL To Final Choice Distribution",
]

fig, axes = plt.subplots(len(substep_order), len(metric_cols), figsize=(5.5 * len(metric_cols), 4.0 * len(substep_order)), sharex=True)

for r, substep_name in enumerate(substep_order):
    part = analysis_substep_df.loc[analysis_substep_df["substep_name"].eq(substep_name)]
    for c, (metric, title) in enumerate(zip(metric_cols, metric_titles)):
        ax = axes[r, c]
        sns.lineplot(
            data=part,
            x="layer_number",
            y=metric,
            hue="trace_bucket",
            hue_order=bucket_order,
            palette=bucket_palette,
            estimator="mean",
            errorbar=None,
            marker="o",
            ax=ax,
            legend=(r == 0 and c == 0),
        )
        ax.set_title(f"{substep_name}: {title}")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean value")
        if not (r == 0 and c == 0) and ax.legend_ is not None:
            ax.legend_.remove()

plt.tight_layout()
plt.show()

In [ ]:
stability_rows = []
for (example_id, readout_method), part in analysis_layer_df.groupby(["example_id", "readout_method"]):
    part = part.sort_values("layer_number")
    pred_seq = part["predicted_choice_idx"].to_numpy()
    layer_seq = part["layer_number"].to_numpy()
    final_pred = int(pred_seq[-1])

    n_prediction_flips = int(np.sum(pred_seq[1:] != pred_seq[:-1]))

    first_match_final_layer = None
    match_idx = np.where(pred_seq == final_pred)[0]
    if len(match_idx):
        first_match_final_layer = int(layer_seq[match_idx[0]])

    stabilization_layer = None
    for i in range(len(pred_seq)):
        if np.all(pred_seq[i:] == final_pred):
            stabilization_layer = int(layer_seq[i])
            break

    stability_rows.append(
        {
            "example_id": example_id,
            "readout_method": readout_method,
            "n_prediction_flips": n_prediction_flips,
            "first_match_final_layer": first_match_final_layer,
            "stabilization_layer": stabilization_layer,
        }
    )

stability_df = (
    pd.DataFrame(stability_rows)
    .merge(clean_small_df[["example_id", "clean_is_correct"]], on="example_id", how="left")
    .merge(trace_bucket_df[["example_id", "trace_bucket"]], on="example_id", how="left")
)
stability_df["final_error"] = (~stability_df["clean_is_correct"]).astype(int)

attention_stability_rows = []
for example_id, part in analysis_attention_df.groupby("example_id"):
    part = part.sort_values("layer_number")
    pred_seq = part["attention_choice_predicted_choice_idx"].to_numpy()
    layer_seq = part["layer_number"].to_numpy()
    final_pred = int(pred_seq[-1])

    n_prediction_flips = int(np.sum(pred_seq[1:] != pred_seq[:-1]))

    first_match_final_layer = None
    match_idx = np.where(pred_seq == final_pred)[0]
    if len(match_idx):
        first_match_final_layer = int(layer_seq[match_idx[0]])

    stabilization_layer = None
    for i in range(len(pred_seq)):
        if np.all(pred_seq[i:] == final_pred):
            stabilization_layer = int(layer_seq[i])
            break

    attention_stability_rows.append(
        {
            "example_id": example_id,
            "attention_n_prediction_flips": n_prediction_flips,
            "attention_first_match_final_layer": first_match_final_layer,
            "attention_stabilization_layer": stabilization_layer,
        }
    )

attention_stability_df = (
    pd.DataFrame(attention_stability_rows)
    .merge(clean_small_df[["example_id", "clean_is_correct"]], on="example_id", how="left")
    .merge(trace_bucket_df[["example_id", "trace_bucket"]], on="example_id", how="left")
)
attention_stability_df["final_error"] = (~attention_stability_df["clean_is_correct"]).astype(int)

display(
    stability_df.groupby(["readout_method", "trace_bucket"])[["n_prediction_flips", "first_match_final_layer", "stabilization_layer"]]
    .agg(["mean", "median", "std"])
    .round(3)
)

## AUC Method

Primary target: `final_error`.

Feature selection rules in the AUC sections:

- normalized entropy versions only
- raw and normalized duplicates are not both used because AUC is invariant to positive linear rescaling
- raw letter-indexed logits and raw letter-indexed attention probabilities are omitted from the generic feature ranking because they are not permutation-invariant across examples
- oracle features are separated from non-oracle features

In [ ]:
def compute_auc_table(frame, feature_cols, group_cols=None):
    rows = []

    if group_cols is None:
        grouped = [((), frame)]
    else:
        grouped = frame.groupby(group_cols, dropna=False)

    for group_key, part in grouped:
        if group_cols is None:
            group_key = ()
        elif not isinstance(group_key, tuple):
            group_key = (group_key,)

        y_all = part["final_error"].to_numpy()

        for feature_name in feature_cols:
            x_all = pd.to_numeric(part[feature_name], errors="coerce").to_numpy(dtype=float)
            mask = np.isfinite(x_all) & np.isfinite(y_all)
            if mask.sum() == 0:
                continue

            y = y_all[mask].astype(int)
            x = x_all[mask]
            if len(np.unique(y)) < 2:
                continue

            error_mean = float(x[y == 1].mean())
            correct_mean = float(x[y == 0].mean())
            direction_toward_error = 1.0 if error_mean >= correct_mean else -1.0
            score = direction_toward_error * x

            row = {
                "feature": feature_name,
                "n_examples": int(mask.sum()),
                "error_prevalence": float(y.mean()),
                "direction_toward_error": int(direction_toward_error),
                "feature_mean_correct": correct_mean,
                "feature_mean_error": error_mean,
                "roc_auc_error": float(roc_auc_score(y, score)),
                "pr_auc_error": float(average_precision_score(y, score)),
            }

            if group_cols is not None:
                for key_name, key_value in zip(group_cols, group_key):
                    row[key_name] = key_value

            rows.append(row)

    return pd.DataFrame(rows)


def plot_auc_trace_panels(df, panel_col, panel_order, feature_order, feature_labels, title_prefix):
    metric_specs = [
        ("roc_auc_error", "ROC AUC", "o", "#1f77b4"),
        ("pr_auc_error", "PR AUC", "s", "#d62728"),
    ]

    fig, axes = plt.subplots(1, len(panel_order), figsize=(7.0 * len(panel_order), 5), sharey=True)
    if len(panel_order) == 1:
        axes = [axes]

    for ax, panel_value in zip(axes, panel_order):
        part = (
            df.loc[df[panel_col].eq(panel_value)]
            .set_index("feature")
            .loc[feature_order]
            .reset_index()
        )
        x = np.arange(len(feature_order))
        error_prevalence = float(part["error_prevalence"].dropna().iloc[0])

        for metric_col, metric_label, marker, color in metric_specs:
            ax.plot(
                x,
                part[metric_col],
                marker=marker,
                markersize=7,
                linewidth=2,
                color=color,
                label=metric_label,
            )

        ax.set_xticks(x)
        ax.set_xticklabels([feature_labels[f] for f in feature_order], rotation=20, ha="right")
        ax.set_title(str(panel_value))
        ax.set_xlabel("Feature")
        ax.set_ylim(0.0, 1.0)
        ax.axhline(0.5, color="black", linewidth=1, linestyle="--", alpha=0.6)
        ax.axhline(error_prevalence, color="#666666", linewidth=1, linestyle=":", alpha=0.9)
        ax.grid(axis="y", alpha=0.3)

    axes[0].set_ylabel("AUC")
    legend_handles = [
        plt.Line2D([0], [0], color="#1f77b4", marker="o", linewidth=2, markersize=7, label="ROC AUC"),
        plt.Line2D([0], [0], color="#d62728", marker="s", linewidth=2, markersize=7, label="PR AUC"),
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="ROC baseline (0.5)"),
        plt.Line2D([0], [0], color="#666666", linestyle=":", linewidth=1, label="PR baseline (error rate)"),
    ]
    axes[0].legend(handles=legend_handles)
    plt.suptitle(title_prefix, y=1.02, fontsize=15)
    plt.tight_layout()
    plt.show()


def plot_auc_heatmaps(df, facet_col, facet_order, feature_order, feature_labels, title_prefix):
    fig, axes = plt.subplots(2, len(facet_order), figsize=(6.5 * len(facet_order), 0.55 * len(feature_order) + 6), sharey=True)
    if len(facet_order) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    metric_info = [
        ("roc_auc_error", "ROC AUC", 0.5, 1.0),
        ("pr_auc_error", "PR AUC", max(0.0, float(df["error_prevalence"].min())), 1.0),
    ]

    for col_idx, facet_value in enumerate(facet_order):
        part = df.loc[df[facet_col].eq(facet_value)].copy()
        for row_idx, (metric_col, metric_title, vmin, vmax) in enumerate(metric_info):
            pivot = (
                part.pivot(index="feature", columns="layer_number", values=metric_col)
                .reindex(index=feature_order)
            )
            pivot.index = [feature_labels[x] for x in pivot.index]
            sns.heatmap(
                pivot,
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
                ax=axes[row_idx, col_idx],
                cbar=(col_idx == len(facet_order) - 1),
            )
            axes[row_idx, col_idx].set_title(f"{facet_value}: {metric_title}")
            axes[row_idx, col_idx].set_xlabel("Layer")
            axes[row_idx, col_idx].set_ylabel("")

    axes[0, 0].set_ylabel("Feature")
    axes[1, 0].set_ylabel("Feature")
    plt.suptitle(title_prefix, y=1.02, fontsize=15)
    plt.tight_layout()
    plt.show()


def plot_auc_layer_lines(df, facet_col, facet_order, feature_order, feature_labels, title_prefix):
    metric_info = [
        ("roc_auc_error", "ROC AUC", 0.5),
        ("pr_auc_error", "PR AUC", max(0.0, float(df["error_prevalence"].min()))),
    ]

    fig, axes = plt.subplots(
        len(metric_info),
        len(facet_order),
        figsize=(6.5 * len(facet_order), 4.8 * len(metric_info)),
        sharex=True,
        sharey="row",
    )
    if len(facet_order) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    palette = sns.color_palette("tab10", n_colors=len(feature_order))

    for col_idx, facet_value in enumerate(facet_order):
        part = df.loc[df[facet_col].eq(facet_value)].copy()
        for row_idx, (metric_col, metric_title, baseline_value) in enumerate(metric_info):
            ax = axes[row_idx, col_idx]
            for color_idx, feature_name in enumerate(feature_order):
                feature_part = part.loc[part["feature"].eq(feature_name)].sort_values("layer_number")
                ax.plot(
                    feature_part["layer_number"],
                    feature_part[metric_col],
                    marker="o",
                    markersize=4,
                    linewidth=2,
                    color=palette[color_idx],
                    label=feature_labels[feature_name],
                )
            ax.axhline(baseline_value, color="black", linewidth=1, linestyle="--", alpha=0.6)
            ax.set_title(f"{facet_value}: {metric_title}")
            ax.set_xlabel("Layer")
            ax.set_ylim(0.0, 1.0)
            ax.grid(axis="y", alpha=0.3)

    axes[0, 0].set_ylabel("AUC")
    axes[1, 0].set_ylabel("AUC")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    legend_map = dict(zip(labels, handles))
    axes[0, 0].legend(legend_map.values(), legend_map.keys(), fontsize=9)
    plt.suptitle(title_prefix, y=1.02, fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
final_layer_trace_df = analysis_layer_df.loc[
    analysis_layer_df["layer_number"].eq(run_config["num_layers"]),
    [
        "example_id",
        "readout_method",
        "best_non_choice_logit",
        "best_choice_minus_best_non_choice_logit",
        "full_vocab_entropy_normalized",
        "answer_choice_entropy_normalized",
        "answer_choice_top1_top2_logit_gap",
        "choice_kl_to_final",
        "prediction_agreement_with_final",
        "true_answer_logit_minus_best_other_choice_logit",
        "true_answer_probability_within_choices",
        "true_answer_rank_within_choices",
        "predicted_choice_is_correct",
    ],
].copy().drop_duplicates(subset=["example_id", "readout_method"])

trace_feature_df = final_layer_trace_df.merge(
    stability_df[["example_id", "readout_method", "n_prediction_flips", "first_match_final_layer", "stabilization_layer", "final_error"]],
    on=["example_id", "readout_method"],
    how="left",
    validate="one_to_one",
)

trace_non_oracle_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
    "choice_kl_to_final",
    "prediction_agreement_with_final",
    "n_prediction_flips",
    "first_match_final_layer",
    "stabilization_layer",
]
trace_oracle_features = [
    "true_answer_logit_minus_best_other_choice_logit",
    "true_answer_probability_within_choices",
    "true_answer_rank_within_choices",
    "predicted_choice_is_correct",
]

trace_auc_non_oracle_df = compute_auc_table(trace_feature_df, trace_non_oracle_features, group_cols=["readout_method"])
trace_auc_oracle_df = compute_auc_table(trace_feature_df, trace_oracle_features, group_cols=["readout_method"])

final_attention_trace_df = analysis_attention_df.loc[
    analysis_attention_df["layer_number"].eq(run_config["num_layers"]),
    [
        "example_id",
        "mean_head_renyi2_entropy_normalized",
        "aggregated_choice_attention_entropy_normalized",
        "aggregated_choice_attention_top1_top2_probability_gap",
        "prediction_agreement_with_final",
        "final_error",
    ],
].copy().drop_duplicates(subset=["example_id"])

final_attention_trace_df = final_attention_trace_df.merge(
    attention_stability_df[["example_id", "attention_n_prediction_flips", "attention_first_match_final_layer", "attention_stabilization_layer"]],
    on="example_id",
    how="left",
    validate="one_to_one",
)
final_attention_trace_df["attention_source"] = "decision_position_attention"

attention_trace_features = [
    "mean_head_renyi2_entropy_normalized",
    "aggregated_choice_attention_entropy_normalized",
    "aggregated_choice_attention_top1_top2_probability_gap",
    "prediction_agreement_with_final",
    "attention_n_prediction_flips",
    "attention_first_match_final_layer",
    "attention_stabilization_layer",
]
attention_trace_auc_df = compute_auc_table(final_attention_trace_df, attention_trace_features, group_cols=["attention_source"])

display(trace_auc_non_oracle_df.sort_values(["readout_method", "roc_auc_error"], ascending=[True, False]).round(4))
display(trace_auc_oracle_df.sort_values(["readout_method", "roc_auc_error"], ascending=[True, False]).round(4))
display(attention_trace_auc_df.sort_values("roc_auc_error", ascending=False).round(4))

In [ ]:
trace_non_oracle_feature_order = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
    "choice_kl_to_final",
    "prediction_agreement_with_final",
    "n_prediction_flips",
    "first_match_final_layer",
    "stabilization_layer",
]
trace_non_oracle_feature_labels = {
    "best_non_choice_logit": "Best non-choice logit",
    "best_choice_minus_best_non_choice_logit": "Best choice minus best non-choice",
    "full_vocab_entropy_normalized": "Full vocab entropy",
    "answer_choice_entropy_normalized": "Choice entropy",
    "answer_choice_top1_top2_logit_gap": "Top1-top2 gap",
    "choice_kl_to_final": "KL to final choice",
    "prediction_agreement_with_final": "Agreement with final prediction",
    "n_prediction_flips": "Prediction flips",
    "first_match_final_layer": "First match final layer",
    "stabilization_layer": "Stabilization layer",
}
plot_auc_trace_panels(
    trace_auc_non_oracle_df,
    panel_col="readout_method",
    panel_order=method_order,
    feature_order=trace_non_oracle_feature_order,
    feature_labels=trace_non_oracle_feature_labels,
    title_prefix="Trace-Level Non-Oracle Features",
)

trace_oracle_feature_order = [
    "true_answer_logit_minus_best_other_choice_logit",
    "true_answer_probability_within_choices",
    "true_answer_rank_within_choices",
    "predicted_choice_is_correct",
]
trace_oracle_feature_labels = {
    "true_answer_logit_minus_best_other_choice_logit": "True answer minus best other",
    "true_answer_probability_within_choices": "True answer probability",
    "true_answer_rank_within_choices": "True answer rank",
    "predicted_choice_is_correct": "Intermediate choice correctness",
}
plot_auc_trace_panels(
    trace_auc_oracle_df,
    panel_col="readout_method",
    panel_order=method_order,
    feature_order=trace_oracle_feature_order,
    feature_labels=trace_oracle_feature_labels,
    title_prefix="Trace-Level Oracle Features",
)

attention_trace_feature_order = [
    "mean_head_renyi2_entropy_normalized",
    "aggregated_choice_attention_entropy_normalized",
    "aggregated_choice_attention_top1_top2_probability_gap",
    "prediction_agreement_with_final",
    "attention_n_prediction_flips",
    "attention_first_match_final_layer",
    "attention_stabilization_layer",
]
attention_trace_feature_labels = {
    "mean_head_renyi2_entropy_normalized": "Mean head 2-Renyi entropy",
    "aggregated_choice_attention_entropy_normalized": "Choice attention entropy",
    "aggregated_choice_attention_top1_top2_probability_gap": "Choice attention top1-top2 gap",
    "prediction_agreement_with_final": "Agreement with final prediction",
    "attention_n_prediction_flips": "Attention prediction flips",
    "attention_first_match_final_layer": "Attention first match final layer",
    "attention_stabilization_layer": "Attention stabilization layer",
}
plot_auc_trace_panels(
    attention_trace_auc_df,
    panel_col="attention_source",
    panel_order=["decision_position_attention"],
    feature_order=attention_trace_feature_order,
    feature_labels=attention_trace_feature_labels,
    title_prefix="Trace-Level Attention Features",
)

In [ ]:
layer_non_oracle_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
    "choice_kl_to_final",
    "prediction_agreement_with_final",
]
layer_oracle_features = [
    "true_answer_logit_minus_best_other_choice_logit",
    "true_answer_probability_within_choices",
    "true_answer_rank_within_choices",
    "predicted_choice_is_correct",
]

layer_auc_non_oracle_df = compute_auc_table(
    analysis_layer_df,
    layer_non_oracle_features,
    group_cols=["readout_method", "layer_number"],
)
layer_auc_oracle_df = compute_auc_table(
    analysis_layer_df,
    layer_oracle_features,
    group_cols=["readout_method", "layer_number"],
)

display(layer_auc_non_oracle_df.head().round(4))
display(layer_auc_oracle_df.head().round(4))

In [ ]:
layer_non_oracle_labels = {
    "best_non_choice_logit": "Best non-choice logit",
    "best_choice_minus_best_non_choice_logit": "Best choice minus best non-choice",
    "full_vocab_entropy_normalized": "Full vocab entropy",
    "answer_choice_entropy_normalized": "Choice entropy",
    "answer_choice_top1_top2_logit_gap": "Top1-top2 gap",
    "choice_kl_to_final": "KL to final choice",
    "prediction_agreement_with_final": "Agreement with final prediction",
}
plot_auc_heatmaps(
    layer_auc_non_oracle_df,
    facet_col="readout_method",
    facet_order=method_order,
    feature_order=layer_non_oracle_features,
    feature_labels=layer_non_oracle_labels,
    title_prefix="Layerwise Non-Oracle Features",
)

layer_oracle_labels = {
    "true_answer_logit_minus_best_other_choice_logit": "True answer minus best other",
    "true_answer_probability_within_choices": "True answer probability",
    "true_answer_rank_within_choices": "True answer rank",
    "predicted_choice_is_correct": "Intermediate choice correctness",
}
plot_auc_heatmaps(
    layer_auc_oracle_df,
    facet_col="readout_method",
    facet_order=method_order,
    feature_order=layer_oracle_features,
    feature_labels=layer_oracle_labels,
    title_prefix="Layerwise Oracle Features",
)

In [ ]:
attention_features = [
    "mean_head_renyi2_entropy_normalized",
    "aggregated_choice_attention_entropy_normalized",
    "aggregated_choice_attention_top1_top2_probability_gap",
    "prediction_agreement_with_final",
]
attention_auc_df = compute_auc_table(
    analysis_attention_df,
    attention_features,
    group_cols=["layer_number"],
)
attention_auc_df["attention_source"] = "decision_position_attention"

attention_feature_labels = {
    "mean_head_renyi2_entropy_normalized": "Mean head 2-Renyi entropy",
    "aggregated_choice_attention_entropy_normalized": "Choice attention entropy",
    "aggregated_choice_attention_top1_top2_probability_gap": "Choice attention top1-top2 gap",
    "prediction_agreement_with_final": "Agreement with final prediction",
}

plot_auc_heatmaps(
    attention_auc_df,
    facet_col="attention_source",
    facet_order=["decision_position_attention"],
    feature_order=attention_features,
    feature_labels=attention_feature_labels,
    title_prefix="Attention Features",
)

In [ ]:
substep_non_oracle_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
    "choice_kl_to_final",
    "prediction_agreement_with_final",
]
substep_oracle_features = [
    "true_answer_logit_minus_best_other_choice_logit",
    "true_answer_probability_within_choices",
    "true_answer_rank_within_choices",
    "predicted_choice_is_correct",
]

substep_auc_non_oracle_df = compute_auc_table(
    analysis_substep_df,
    substep_non_oracle_features,
    group_cols=["substep_name", "layer_number"],
)
substep_auc_oracle_df = compute_auc_table(
    analysis_substep_df,
    substep_oracle_features,
    group_cols=["substep_name", "layer_number"],
)

substep_order = ["pre_attn", "post_attn", "post_mlp"]
substep_non_oracle_labels = {
    "best_non_choice_logit": "Best non-choice logit",
    "best_choice_minus_best_non_choice_logit": "Best choice minus best non-choice",
    "full_vocab_entropy_normalized": "Full vocab entropy",
    "answer_choice_entropy_normalized": "Choice entropy",
    "answer_choice_top1_top2_logit_gap": "Top1-top2 gap",
    "choice_kl_to_final": "KL to final choice",
    "prediction_agreement_with_final": "Agreement with final prediction",
}
substep_oracle_labels = {
    "true_answer_logit_minus_best_other_choice_logit": "True answer minus best other",
    "true_answer_probability_within_choices": "True answer probability",
    "true_answer_rank_within_choices": "True answer rank",
    "predicted_choice_is_correct": "Intermediate choice correctness",
}

plot_auc_heatmaps(
    substep_auc_non_oracle_df,
    facet_col="substep_name",
    facet_order=substep_order,
    feature_order=substep_non_oracle_features,
    feature_labels=substep_non_oracle_labels,
    title_prefix="Substep Non-Oracle Features",
)
plot_auc_heatmaps(
    substep_auc_oracle_df,
    facet_col="substep_name",
    facet_order=substep_order,
    feature_order=substep_oracle_features,
    feature_labels=substep_oracle_labels,
    title_prefix="Substep Oracle Features",
)

## Steering-Usable Predictor View

This section keeps only features available at the intervention point.

Excluded from this view:
- true-answer features because they require oracle labels
- trajectory summaries such as prediction flips and stabilization layer because they require future layers
- agreement with final prediction and KL to final choice because they depend on the final-layer output

In [ ]:
steering_layer_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
]
steering_layer_feature_labels = {
    "best_non_choice_logit": "Best non-choice logit",
    "best_choice_minus_best_non_choice_logit": "Best choice minus best non-choice",
    "full_vocab_entropy_normalized": "Full vocab entropy",
    "answer_choice_entropy_normalized": "Choice entropy",
    "answer_choice_top1_top2_logit_gap": "Top1-top2 gap",
}

steering_layer_auc_df = layer_auc_non_oracle_df.loc[
    layer_auc_non_oracle_df["feature"].isin(steering_layer_features)
].copy()

plot_auc_layer_lines(
    steering_layer_auc_df,
    facet_col="readout_method",
    facet_order=method_order,
    feature_order=steering_layer_features,
    feature_labels=steering_layer_feature_labels,
    title_prefix="Layerwise Steering-Usable Features",
)

display(
    steering_layer_auc_df.sort_values(
        ["readout_method", "pr_auc_error", "roc_auc_error"],
        ascending=[True, False, False],
    ).round(4)
)

In [ ]:
steering_attention_features = [
    "mean_head_renyi2_entropy_normalized",
    "aggregated_choice_attention_entropy_normalized",
    "aggregated_choice_attention_top1_top2_probability_gap",
]
steering_attention_feature_labels = {
    "mean_head_renyi2_entropy_normalized": "Mean head 2-Renyi entropy",
    "aggregated_choice_attention_entropy_normalized": "Choice attention entropy",
    "aggregated_choice_attention_top1_top2_probability_gap": "Choice attention top1-top2 gap",
}
steering_attention_auc_df = attention_auc_df.loc[
    attention_auc_df["feature"].isin(steering_attention_features)
].copy()

plot_auc_layer_lines(
    steering_attention_auc_df,
    facet_col="attention_source",
    facet_order=["decision_position_attention"],
    feature_order=steering_attention_features,
    feature_labels=steering_attention_feature_labels,
    title_prefix="Attention Steering-Usable Features",
)

display(
    steering_attention_auc_df.sort_values(
        ["pr_auc_error", "roc_auc_error"],
        ascending=[False, False],
    ).round(4)
)

In [ ]:
steering_substep_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
]
steering_substep_feature_labels = {
    "best_non_choice_logit": "Best non-choice logit",
    "best_choice_minus_best_non_choice_logit": "Best choice minus best non-choice",
    "full_vocab_entropy_normalized": "Full vocab entropy",
    "answer_choice_entropy_normalized": "Choice entropy",
    "answer_choice_top1_top2_logit_gap": "Top1-top2 gap",
}
substep_order = ["pre_attn", "post_attn", "post_mlp"]
steering_substep_auc_df = substep_auc_non_oracle_df.loc[
    substep_auc_non_oracle_df["feature"].isin(steering_substep_features)
].copy()

plot_auc_layer_lines(
    steering_substep_auc_df,
    facet_col="substep_name",
    facet_order=substep_order,
    feature_order=steering_substep_features,
    feature_labels=steering_substep_feature_labels,
    title_prefix="Substep Steering-Usable Features",
)

display(
    steering_substep_auc_df.sort_values(
        ["substep_name", "pr_auc_error", "roc_auc_error"],
        ascending=[True, False, False],
    ).round(4)
)

## Train-Split Sparse Logistic Regression

One logistic regression model is fit separately at each layer.

Features included:
- direct-readout steering-usable features
- tuned-lens steering-usable features
- compact attention steering-usable features when train attention data is available

Regularization:
- L1 penalty with cross-validated strength selection
- coefficients driven to zero are treated as unused features

Decision rule:
- the goal is conservative bad-trace flagging
- a trace should be flagged only when the estimated error probability is high enough to satisfy a strict precision target
- false positives on good traces are treated as more costly than missed bad traces

In [ ]:
steering_base_features = [
    "best_non_choice_logit",
    "best_choice_minus_best_non_choice_logit",
    "full_vocab_entropy_normalized",
    "answer_choice_entropy_normalized",
    "answer_choice_top1_top2_logit_gap",
]

direct_train_model_df = (
    model_analysis_layer_df.loc[
        model_analysis_layer_df["readout_method"].eq("direct_readout"),
        ["example_id", "layer_number", "final_error", *steering_base_features],
    ]
    .rename(columns={feature: f"direct__{feature}" for feature in steering_base_features})
    .reset_index(drop=True)
)

tuned_train_model_df = (
    model_analysis_layer_df.loc[
        model_analysis_layer_df["readout_method"].eq("tuned_lens"),
        ["example_id", "layer_number", *steering_base_features],
    ]
    .rename(columns={feature: f"tuned__{feature}" for feature in steering_base_features})
    .reset_index(drop=True)
)

train_model_df = direct_train_model_df.merge(
    tuned_train_model_df,
    on=["example_id", "layer_number"],
    how="inner",
    validate="one_to_one",
)

attention_train_feature_cols = [
    "mean_head_renyi2_entropy_normalized",
    "aggregated_choice_attention_entropy_normalized",
    "aggregated_choice_attention_top1_top2_probability_gap",
]
attention_train_model_df = (
    model_attention_analysis_df[
        ["example_id", "layer_number", *attention_train_feature_cols]
    ]
    .rename(columns={feature: f"attention__{feature}" for feature in attention_train_feature_cols})
    .reset_index(drop=True)
)
train_model_df = train_model_df.merge(
    attention_train_model_df,
    on=["example_id", "layer_number"],
    how="left",
    validate="one_to_one",
)

train_model_feature_cols = (
    [f"direct__{feature}" for feature in steering_base_features]
    + [f"tuned__{feature}" for feature in steering_base_features]
    + [f"attention__{feature}" for feature in attention_train_feature_cols]
)

train_model_df = train_model_df.dropna(subset=train_model_feature_cols).reset_index(drop=True)

display(train_model_df.head())
print("n_train_rows:", len(train_model_df))
print("n_model_features:", len(train_model_feature_cols))
print("model_split_prefix:", repr(MODEL_SPLIT_PREFIX))
print("model_error_rate:", round(train_model_df["final_error"].mean(), 4))

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
regularization_grid = np.logspace(-3, 2, 12)

train_layer_model_rows = []
train_layer_coef_rows = []
train_layer_prediction_rows = []

for layer_number, part in train_model_df.groupby("layer_number"):
    X = part[train_model_feature_cols].to_numpy(dtype=float)
    y = part["final_error"].to_numpy(dtype=int)

    model_pipeline = make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=regularization_grid,
            cv=5,
            penalty="l1",
            solver="saga",
            scoring="average_precision",
            max_iter=5000,
            n_jobs=-1,
            random_state=42,
            refit=True,
        ),
    )

    oof_error_probability = cross_val_predict(
        model_pipeline,
        X,
        y,
        cv=outer_cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]

    model_pipeline.fit(X, y)
    fitted_model = model_pipeline.named_steps["logisticregressioncv"]
    fitted_coef = fitted_model.coef_.ravel()
    nonzero_mask = fitted_coef != 0
    selected_c = float(np.ravel(fitted_model.C_)[0])

    train_layer_model_rows.append(
        {
            "layer_number": int(layer_number),
            "n_examples": int(len(part)),
            "error_rate": float(y.mean()),
            "selected_c": selected_c,
            "cv_roc_auc_error": float(roc_auc_score(y, oof_error_probability)),
            "cv_pr_auc_error": float(average_precision_score(y, oof_error_probability)),
            "nonzero_feature_count": int(nonzero_mask.sum()),
            "zeroed_feature_count": int((~nonzero_mask).sum()),
        }
    )

    for example_id, probability, target in zip(part["example_id"], oof_error_probability, y):
        train_layer_prediction_rows.append(
            {
                "example_id": example_id,
                "layer_number": int(layer_number),
                "final_error": int(target),
                "predicted_error_probability": float(probability),
            }
        )

    for feature_name, coefficient in zip(train_model_feature_cols, fitted_coef):
        train_layer_coef_rows.append(
            {
                "layer_number": int(layer_number),
                "feature": feature_name,
                "coefficient": float(coefficient),
                "abs_coefficient": float(abs(coefficient)),
                "is_nonzero": bool(coefficient != 0),
            }
        )

train_layer_model_df = pd.DataFrame(train_layer_model_rows).sort_values("layer_number").reset_index(drop=True)
train_layer_coef_df = pd.DataFrame(train_layer_coef_rows)
train_layer_prediction_df = pd.DataFrame(train_layer_prediction_rows)

predictor_export_dir = RUN_DIR / "predictor_analysis_exports"
predictor_export_dir.mkdir(parents=True, exist_ok=True)
predictor_export_stem = f"{MODEL_SPLIT_PREFIX}layerwise_logreg"

train_layer_model_df.to_parquet(predictor_export_dir / f"{predictor_export_stem}_model_summary.parquet", index=False)
train_layer_coef_df.to_parquet(predictor_export_dir / f"{predictor_export_stem}_coefficients.parquet", index=False)
train_layer_prediction_df.to_parquet(predictor_export_dir / f"{predictor_export_stem}_oof_predictions.parquet", index=False)

print("predictor_export_dir:", predictor_export_dir)
display(train_layer_model_df.round(4))

In [ ]:
PRECISION_TARGETS = [0.80, 0.90, 0.95]

def compute_threshold_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    flagged_count = int(y_pred.sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    flag_rate = flagged_count / len(y_true) if len(y_true) > 0 else np.nan

    return {
        "threshold": float(threshold),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "flagged_count": flagged_count,
        "precision": precision,
        "recall": recall,
        "false_positive_rate": false_positive_rate,
        "flag_rate": flag_rate,
    }


threshold_metric_rows = []
precision_target_rows = []

for layer_number, part in train_layer_prediction_df.groupby("layer_number"):
    y_true = part["final_error"].to_numpy(dtype=int)
    y_prob = part["predicted_error_probability"].to_numpy(dtype=float)
    thresholds = np.sort(np.unique(y_prob))[::-1]

    layer_metrics = []
    for threshold in thresholds:
        metrics = compute_threshold_metrics(y_true, y_prob, threshold)
        metrics["layer_number"] = int(layer_number)
        layer_metrics.append(metrics)
        threshold_metric_rows.append(metrics)

    layer_metric_df = pd.DataFrame(layer_metrics)

    for precision_target in PRECISION_TARGETS:
        candidate_df = layer_metric_df.loc[
            (layer_metric_df["precision"] >= precision_target)
            & (layer_metric_df["flagged_count"] > 0)
        ].copy()

        if candidate_df.empty:
            precision_target_rows.append(
                {
                    "layer_number": int(layer_number),
                    "precision_target": float(precision_target),
                    "threshold": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "false_positive_rate": np.nan,
                    "flag_rate": np.nan,
                    "flagged_count": 0,
                    "tp": 0,
                    "fp": 0,
                    "tn": int((y_true == 0).sum()),
                    "fn": int((y_true == 1).sum()),
                }
            )
            continue

        chosen_row = candidate_df.sort_values(
            ["recall", "flagged_count", "precision", "false_positive_rate"],
            ascending=[False, False, False, True],
        ).iloc[0]

        precision_target_rows.append(
            {
                "layer_number": int(layer_number),
                "precision_target": float(precision_target),
                "threshold": float(chosen_row["threshold"]),
                "precision": float(chosen_row["precision"]),
                "recall": float(chosen_row["recall"]),
                "false_positive_rate": float(chosen_row["false_positive_rate"]),
                "flag_rate": float(chosen_row["flag_rate"]),
                "flagged_count": int(chosen_row["flagged_count"]),
                "tp": int(chosen_row["tp"]),
                "fp": int(chosen_row["fp"]),
                "tn": int(chosen_row["tn"]),
                "fn": int(chosen_row["fn"]),
            }
        )

train_threshold_metric_df = pd.DataFrame(threshold_metric_rows)
train_precision_target_df = pd.DataFrame(precision_target_rows)

train_threshold_metric_df.to_parquet(predictor_export_dir / f"{predictor_export_stem}_threshold_metrics.parquet", index=False)
train_precision_target_df.to_parquet(predictor_export_dir / f"{predictor_export_stem}_precision_targets.parquet", index=False)

display(train_precision_target_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharex=True)

sns.lineplot(
    data=train_precision_target_df,
    x="layer_number",
    y="precision",
    hue="precision_target",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Achieved Precision By Layer")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Precision")
axes[0].set_ylim(0.0, 1.0)

sns.lineplot(
    data=train_precision_target_df,
    x="layer_number",
    y="recall",
    hue="precision_target",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Recall Under Precision Constraint")
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Recall")
axes[1].set_ylim(0.0, 1.0)

sns.lineplot(
    data=train_precision_target_df,
    x="layer_number",
    y="false_positive_rate",
    hue="precision_target",
    marker="o",
    ax=axes[2],
)
axes[2].set_title("False Positive Rate By Layer")
axes[2].set_xlabel("Layer")
axes[2].set_ylabel("False positive rate")
axes[2].set_ylim(0.0, 1.0)

sns.lineplot(
    data=train_precision_target_df,
    x="layer_number",
    y="flag_rate",
    hue="precision_target",
    marker="o",
    ax=axes[3],
)
axes[3].set_title("Flag Rate By Layer")
axes[3].set_xlabel("Layer")
axes[3].set_ylabel("Flag rate")
axes[3].set_ylim(0.0, 1.0)

for ax in axes[1:]:
    if ax.legend_ is not None:
        ax.legend_.remove()

plt.tight_layout()
plt.show()

OPERATING_PRECISION_TARGET = 0.90
top_operating_layers_df = (
    train_precision_target_df.loc[train_precision_target_df["precision_target"].eq(OPERATING_PRECISION_TARGET)]
    .sort_values(["recall", "flagged_count", "false_positive_rate"], ascending=[False, False, True])
    .reset_index(drop=True)
)

display(top_operating_layers_df.round(4))

In [ ]:
operating_fire_df = (
    train_precision_target_df.loc[
        train_precision_target_df["precision_target"].eq(OPERATING_PRECISION_TARGET)
    ]
    .copy()
    .sort_values(["flagged_count", "flag_rate", "recall"], ascending=[False, False, False])
    .reset_index(drop=True)
)

operating_fire_summary_df = operating_fire_df[[
    "flagged_count",
    "flag_rate",
    "precision",
    "recall",
    "false_positive_rate",
]].describe().round(4)

display(operating_fire_df[[
    "layer_number",
    "threshold",
    "flagged_count",
    "flag_rate",
    "precision",
    "recall",
    "false_positive_rate",
    "tp",
    "fp",
    "tn",
    "fn",
]].round(4))

display(operating_fire_summary_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=False)

sns.barplot(
    data=operating_fire_df,
    x="layer_number",
    y="flagged_count",
    color="#e76f51",
    ax=axes[0],
)
axes[0].set_title(f"Flagged Count By Layer | Precision Target {OPERATING_PRECISION_TARGET:.2f}")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Flagged count")

sns.histplot(
    data=operating_fire_df,
    x="flagged_count",
    bins=min(20, max(5, operating_fire_df["flagged_count"].nunique())),
    color="#264653",
    ax=axes[1],
)
axes[1].set_title(f"Distribution Of Flagged Counts | Precision Target {OPERATING_PRECISION_TARGET:.2f}")
axes[1].set_xlabel("Flagged count")
axes[1].set_ylabel("Number of layers")

plt.tight_layout()
plt.show()

In [ ]:
CONFUSION_PRECISION_TARGET = 0.90
TOP_K_CONFUSION_LAYERS = 6

confusion_layer_df = (
    train_precision_target_df.loc[
        train_precision_target_df["precision_target"].eq(CONFUSION_PRECISION_TARGET)
        & train_precision_target_df["threshold"].notna()
    ]
    .sort_values(["recall", "flagged_count", "false_positive_rate"], ascending=[False, False, True])
    .head(TOP_K_CONFUSION_LAYERS)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, len(confusion_layer_df), figsize=(4.5 * len(confusion_layer_df), 4.5))
if len(confusion_layer_df) == 1:
    axes = [axes]

for ax, (_, summary_row) in zip(axes, confusion_layer_df.iterrows()):
    layer_number = int(summary_row["layer_number"])
    threshold = float(summary_row["threshold"])
    part = train_layer_prediction_df.loc[train_layer_prediction_df["layer_number"].eq(layer_number)]
    y_true = part["final_error"].to_numpy(dtype=int)
    y_pred = (part["predicted_error_probability"].to_numpy(dtype=float) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Predicted correct", "Predicted error"],
    )
    disp.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(
        f"Layer {layer_number}\n"
        f"precision={summary_row['precision']:.2f}, recall={summary_row['recall']:.2f}\n"
        f"fpr={summary_row['false_positive_rate']:.2f}, flag_rate={summary_row['flag_rate']:.2f}"
    )

plt.tight_layout()
plt.show()

In [ ]:
coefficient_pivot = (
    train_layer_coef_df
    .pivot(index="feature", columns="layer_number", values="coefficient")
    .reindex(index=train_model_feature_cols)
)

plt.figure(figsize=(16, 0.45 * len(train_model_feature_cols) + 4))
sns.heatmap(
    coefficient_pivot,
    cmap="coolwarm",
    center=0.0,
)
plt.title("Sparse Logistic Regression Coefficients By Layer")
plt.xlabel("Layer")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(
    train_layer_coef_df.loc[train_layer_coef_df["is_nonzero"]]
    .sort_values(["layer_number", "abs_coefficient"], ascending=[True, False])
    .groupby("layer_number")
    .head(8)
    .round(4)
)